# Deep Hedging - LSTM Neural Network Training

This notebook trains an LSTM neural network model to perform optimal delta hedging on option paths under transaction costs. The LSTM structure helps the model exploit sequential dependency and memory from path histories (spot changes, past delta updates). It exports the final model to a JIT TorchScript format (`models/deep_hedge_lstm.pt`).

In [ ]:
import os
import glob
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from datetime import datetime, timezone

In [ ]:
device = torch.device("cpu")
print(f"Using device: {device}")

## 1. Helper Functions & Data Loading

In [ ]:
def calculate_cvar_loss(pnl, alpha=0.05):
    sorted_pnl, _ = torch.sort(pnl)
    num_samples = sorted_pnl.size(0)
    cvar_index = int(np.floor(num_samples * alpha))
    if cvar_index < 1:
        cvar_index = 1
    worst_pnl = sorted_pnl[:cvar_index]
    cvar_loss = -torch.mean(worst_pnl)
    return cvar_loss

def parse_occ_symbol(symbol):
    match = re.search(r"(\d{2})(\d{2})(\d{2})([CP])(\d{8})$", symbol)
    if not match:
        # Fallback to NIFTY style
        nifty_match = re.search(r"NIFTY(\d{2})([0-9A-Z]+)(\d{5})(CE|PE)$", symbol)
        if nifty_match:
            yy, date_part, strike_raw, cp = nifty_match.groups()
            expiry = datetime(2026, 2, 26, 16, 0, tzinfo=timezone.utc)
            strike = float(strike_raw)
            opt_type = 'CALL' if cp == 'CE' else 'PUT'
            return expiry, strike, opt_type
        return None, None, None
    yy, mm, dd, cp, strike_raw = match.groups()
    year = 2000 + int(yy) if int(yy) < 80 else 1900 + int(yy)
    expiry = datetime(year, int(mm), int(dd), 16, 0, tzinfo=timezone.utc)
    strike = int(strike_raw) / 1000.0
    opt_type = 'CALL' if cp == 'C' else 'PUT'
    return expiry, strike, opt_type

def implied_volatility_approx(spot, strike, dte, price, is_call=True):
    if dte <= 0.0 or spot <= 0.0 or price <= 0.0:
        return 0.20
    try:
        val = (price / spot) * np.sqrt(2 * np.pi / dte)
        return float(np.clip(val, 0.05, 0.80))
    except:
        return 0.20

def generate_synthetic_paths(num_paths=2000, path_len=30):
    import math
    print(f"Generating {num_paths} synthetic GBM paths...")
    paths_collected = []
    dt = 1.0 / 252.0
    for _ in range(num_paths):
        S0 = np.random.uniform(50.0, 500.0)
        strike = S0 * np.random.uniform(0.9, 1.1)
        iv = np.random.uniform(0.1, 0.5)
        dte0 = np.random.uniform(10.0, 60.0) / 365.25
        
        S_seq = [S0]
        for t in range(path_len):
            shock = np.random.normal()
            dS = S_seq[-1] * (0.05 * dt + iv * np.sqrt(dt) * shock)
            S_seq.append(max(1.0, S_seq[-1] + dS))
            
        C_seq = []
        DTE_seq = []
        IV_seq = []
        for t in range(path_len + 1):
            dte_t = max(0.0001, dte0 - t * dt)
            DTE_seq.append(dte_t)
            d1 = (math.log(S_seq[t] / strike) + (0.05 + 0.5 * iv**2) * dte_t) / (iv * math.sqrt(dte_t))
            d2 = d1 - iv * math.sqrt(dte_t)
            cdf_d1 = 0.5 * (1.0 + math.erf(d1 / math.sqrt(2.0)))
            cdf_d2 = 0.5 * (1.0 + math.erf(d2 / math.sqrt(2.0)))
            c_price = S_seq[t] * cdf_d1 - strike * math.exp(-0.05 * dte_t) * cdf_d2
            C_seq.append(max(0.01, c_price))
            IV_seq.append(iv)
            
        paths_collected.append({
            'S': torch.tensor(S_seq, dtype=torch.float32),
            'C': torch.tensor(C_seq, dtype=torch.float32),
            'IV': torch.tensor(IV_seq, dtype=torch.float32),
            'DTE': torch.tensor(DTE_seq, dtype=torch.float32),
            'strike': strike
        })
    return paths_collected

def load_real_paths():
    print("Loading option parquet and CSV data...")
    paths_collected = []
    
    # 1. Load Parquet files from data directory recursively
    data_root = "c:\\Users\\User\\Desktop\\Affinity-Core\\data"
    if os.path.exists(data_root):
        ticker_dirs = [os.path.join(data_root, d) for d in os.listdir(data_root) if os.path.isdir(os.path.join(data_root, d)) and d != "temp"]
        for t_dir in ticker_dirs:
            parquet_files = []
            for root, _, files in os.walk(t_dir):
                for f in files:
                    if f.endswith(".parquet"):
                        parquet_files.append(os.path.join(root, f))
            # Sort and sample to prevent overflow
            parquet_files = sorted(parquet_files)[:20]
            for f_path in parquet_files:
                try:
                    df = pd.read_parquet(f_path)
                    unique_syms = df['symbol'].unique()
                    for sym_raw in unique_syms[:100]: # limit per file
                        sym = str(sym_raw)
                        expiry, strike, opt_type = parse_occ_symbol(sym)
                        if not expiry or strike is None:
                            continue
                        contract_df = df[df['symbol'] == sym_raw].sort_values('ts_recv')
                        if len(contract_df) < 31:
                            continue
                        spots = ((contract_df['underlying_bid_px'] + contract_df['underlying_ask_px']) / 2.0).values
                        opt_prices = ((contract_df['bid_px'] + contract_df['ask_px']) / 2.0).values
                        ts_recvs = contract_df['ts_recv'].values
                        
                        path_len = 30
                        for offset in range(0, len(spots) - path_len - 1, 5):
                            S_seq = spots[offset : offset + path_len + 1]
                            C_seq = opt_prices[offset : offset + path_len + 1]
                            t_seq = ts_recvs[offset : offset + path_len + 1]
                            
                            if np.any(S_seq <= 0.0) or np.any(C_seq <= 0.0):
                                continue
                                
                            IV_seq = []
                            DTE_seq = []
                            for t_idx in range(path_len + 1):
                                curr_dt = pd.to_datetime(t_seq[t_idx])
                                if curr_dt.tzinfo is None:
                                    curr_dt = curr_dt.tz_localize(timezone.utc)
                                else:
                                    curr_dt = curr_dt.tz_convert(timezone.utc)
                                time_diff = expiry - curr_dt
                                curr_dte = max(0.0001, time_diff.total_seconds() / (365.25 * 24 * 3600))
                                DTE_seq.append(curr_dte)
                                IV_seq.append(implied_volatility_approx(S_seq[t_idx], strike, curr_dte, C_seq[t_idx]))
                                
                            paths_collected.append({
                                'S': torch.tensor(S_seq, dtype=torch.float32),
                                'C': torch.tensor(C_seq, dtype=torch.float32),
                                'IV': torch.tensor(IV_seq, dtype=torch.float32),
                                'DTE': torch.tensor(DTE_seq, dtype=torch.float32),
                                'strike': strike
                            })
                except Exception as e:
                    print(f"Error loading parquet {f_path}: {e}")
                    
    # 2. Load CSV files from root directory
    root_dir = "c:\\Users\\User\\Desktop\\Affinity-Core"
    csv_files = [os.path.join(root_dir, f) for f in os.listdir(root_dir) if f.endswith(".csv") and "option_minute_prices" in f]
    for csv_path in csv_files:
        try:
            print(f"Loading options CSV {csv_path}...")
            df = pd.read_csv(csv_path)
            # Find the underlying price column
            fut_sym = [s for s in df['symbol'].unique() if "FUT" in str(s)]
            if not fut_sym:
                continue
            fut_df = df[df['symbol'] == fut_sym[0]].sort_values('minute_end')
            spots_map = dict(zip(fut_df['minute_end'], fut_df['last_trade_price']))
            
            opt_symbols = [s for s in df['symbol'].unique() if "FUT" not in str(s)]
            for sym in opt_symbols[:100]:
                expiry, strike, opt_type = parse_occ_symbol(str(sym))
                if not expiry or strike is None:
                    continue
                contract_df = df[df['symbol'] == sym].sort_values('minute_end')
                if len(contract_df) < 31:
                    continue
                spots = []
                opt_prices = []
                # Reconstruct matching timestamps
                for _, row in contract_df.iterrows():
                    m = row['minute_end']
                    if m in spots_map:
                        spots.append(spots_map[m])
                        opt_prices.append(row['last_trade_price'])
                if len(spots) < 31:
                    continue
                
                path_len = 30
                for offset in range(0, len(spots) - path_len - 1, 5):
                    S_seq = spots[offset : offset + path_len + 1]
                    C_seq = opt_prices[offset : offset + path_len + 1]
                    
                    IV_seq = []
                    DTE_seq = []
                    for t_idx in range(path_len + 1):
                        curr_dte = max(0.0001, (22.0 - t_idx) / 365.25) # Mock DTE
                        DTE_seq.append(curr_dte)
                        IV_seq.append(implied_volatility_approx(S_seq[t_idx], strike, curr_dte, C_seq[t_idx]))
                        
                    paths_collected.append({
                        'S': torch.tensor(S_seq, dtype=torch.float32),
                        'C': torch.tensor(C_seq, dtype=torch.float32),
                        'IV': torch.tensor(IV_seq, dtype=torch.float32),
                        'DTE': torch.tensor(DTE_seq, dtype=torch.float32),
                        'strike': strike
                    })
        except Exception as e:
            print(f"Error loading CSV {csv_path}: {e}")
            
    print(f"Successfully constructed {len(paths_collected)} real option paths!")
    return paths_collected


## 2. LSTM Model Definition

In [ ]:
class LSTMHedger(nn.Module):
    def __init__(self, input_dim=5, hidden_dim=32):
        super(LSTMHedger, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True, num_layers=2, dropout=0.1)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )
        
    def forward(self, x):
        # x shape: (batch_size, sequence_length, features) -> (B, seq, 5)
        x_norm = x.clone()
        x_norm[..., 0] = x[..., 1] + 1.0
        out, _ = self.lstm(x_norm)
        # Use final sequence output, map to delta [0, 1] using Sigmoid
        return torch.sigmoid(self.fc(out[:, -1, :]))

## 3. Data Loading & Model Initialization

In [ ]:
paths = load_real_paths()
synthetic_paths = generate_synthetic_paths(num_paths=1500)
paths.extend(synthetic_paths)
assert len(paths) > 0, "No paths were loaded. Please check your data folders!"

S_tensor = torch.stack([p['S'] for p in paths])
C_tensor = torch.stack([p['C'] for p in paths])
IV_tensor = torch.stack([p['IV'] for p in paths])
DTE_tensor = torch.stack([p['DTE'] for p in paths])
K_tensor = torch.tensor([p['strike'] for p in paths], dtype=torch.float32).unsqueeze(1)
print(f"Total dataset size (real + synthetic): {len(paths)}")


## 4. Training Loop (CVaR Tail-Risk Optimization)

In [ ]:
epochs = 40
lr = 0.01
cost_rate = 0.0005 # 5 bps transaction costs
num_steps = 30
batch_size = 512
num_samples = len(paths)

lstm_model = LSTMHedger(input_dim=5)
optimizer = optim.Adam(lstm_model.parameters(), lr=lr)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

for epoch in range(epochs):
    lstm_model.train()
    epoch_loss = 0.0
    indices = torch.randperm(num_samples)
    
    for idx in range(0, num_samples, batch_size):
        batch_idx = indices[idx : idx + batch_size]
        b_S = S_tensor[batch_idx]
        b_C = C_tensor[batch_idx]
        b_IV = IV_tensor[batch_idx]
        b_DTE = DTE_tensor[batch_idx]
        b_K = K_tensor[batch_idx]
        
        b_size = len(batch_idx)
        deltas = torch.zeros(b_size, num_steps + 1)
        delta_prev = torch.zeros(b_size, 1)
        
        for t in range(num_steps):
            S_t = b_S[:, t:t+1]
            strike_dist = (S_t / b_K) - 1.0
            dte_t = b_DTE[:, t:t+1]
            iv_t = b_IV[:, t:t+1]
            
            # Input shape for LSTM: (batch_size, sequence_length, features) -> (B, 1, 5)
            features_t = torch.cat([S_t, strike_dist, dte_t, iv_t, delta_prev], dim=1).unsqueeze(1)
            delta_t = lstm_model(features_t)
            deltas[:, t] = delta_t.squeeze(1)
            delta_prev = delta_t.detach()
            
        opt_payout = b_C[:, -1] - b_C[:, 0]
        price_changes = b_S[:, 1:] - b_S[:, :-1]
        trading_pnl = torch.sum(deltas[:, :-1] * price_changes, dim=1)
        
        trade_turnover = torch.abs(deltas[:, 1:] - deltas[:, :-1])
        transaction_costs = torch.sum(trade_turnover * b_S[:, :-1] * cost_rate, dim=1)
        
        pnl = -opt_payout + trading_pnl - transaction_costs
        loss = calculate_cvar_loss(pnl)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * b_size
        
    scheduler.step()
    print(f"Epoch {epoch+1}/{epochs}, Loss (CVaR): {epoch_loss / num_samples:.4f}")

## 5. Export TorchScript traced module

In [ ]:
os.makedirs("../models", exist_ok=True)

lstm_model.eval()
traced_lstm = torch.jit.trace(lstm_model, torch.randn(2, 1, 5))

# Export JIT models to consolidated project root folder
traced_lstm.save("../models/deep_hedge_lstm.pt")
print("LSTM Model JIT traced and exported successfully to models/deep_hedge_lstm.pt!")